# 🔬 Standalone Out-of-Distribution & Tiled Inference Evaluation
This notebook evaluates trained YOLOv11n-seg checkpoints on **unseen, full-resolution uncropped road crack photos**:
1. **Direct Resizing Evaluation**: Standard evaluation at $512 \times 512$.
2. **Gaussian-Weighted Tiled / Sliding-Window Inference**: Dividing $2000 \times 1500$ uncropped images into overlapping $512 \times 512$ patches ($25\%$ overlap, $384\text{px}$ stride) with **2D Gaussian Apodization Blending** (eliminating border artifacts and weighting center predictions).
3. **Head-to-Head Comparison**: Compares all available checkpoints (Baseline, Mask KD, Foreground-Dilated, LayerKD, Focal, Combined).


In [1]:
# ── Environment & Imports ──
!mkdir -p scripts configs utils distillation data/datasets results
!pip install -q ultralytics albumentations pycocotools opencv-python Pillow matplotlib tqdm pandas
import os, cv2, json, time, glob
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from ultralytics import YOLO


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 2.1 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


In [2]:
%%writefile scripts/convert_crack500_uncropped.py
#!/usr/bin/env python3
"""
Crack500 Uncropped Test/Val → YOLO seg format converter
======================================================
Converts the original uncropped validation and test sets of Crack500.
Handles EXIF orientation for images by rotating the corresponding masks.

Source directories:
  data/datasets/crack500/valdata/   ← contains {stem}.jpg and {stem}_mask.png
  data/datasets/crack500/testdata/  ← contains {stem}.jpg and {stem}_mask.png

Output (YOLO seg format):
  data/datasets/crack500_uncropped_yolo/
  ├── images/
  │   ├── val/
  │   └── test/
  ├── labels/
  │   ├── val/
  │   └── test/
  └── dataset.yaml
"""

import os
import cv2
import numpy as np
import argparse
import shutil
from pathlib import Path
from tqdm import tqdm
from PIL import Image


CLASS_ID = 0        # single class: crack
MIN_AREA = 50       # minimum pixel area to keep an instance
MIN_POINTS = 6      # minimum polygon points (3 coordinate pairs)


def get_exif_rotation(img_path: Path):
    """Retrieve EXIF orientation tag from image."""
    try:
        with Image.open(img_path) as im:
            exif = im.getexif()
            if exif:
                return exif.get(274)  # 274 is the Orientation tag
    except Exception:
        pass
    return None


def rotate_mask_to_match_image(mask: np.ndarray, exif_orientation: int) -> np.ndarray:
    """Rotate mask array to match image rotation applied by cv2.imread based on EXIF."""
    if exif_orientation == 6:
        return cv2.rotate(mask, cv2.ROTATE_90_CLOCKWISE)
    elif exif_orientation == 8:
        return cv2.rotate(mask, cv2.ROTATE_90_COUNTERCLOCKWISE)
    elif exif_orientation == 3:
        return cv2.rotate(mask, cv2.ROTATE_180)
    return mask


def binary_mask_to_yolo_instances(mask_path: str, img_w: int, img_h: int, exif_orientation: int = None) -> list[str]:
    """
    Read binary PNG mask → rotate based on EXIF → split into instances via connectedComponents
    → convert each to normalized YOLO seg polygon string.
    """
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return []

    if exif_orientation:
        mask = rotate_mask_to_match_image(mask, exif_orientation)

    # Threshold (Crack500 masks are binary 0/255)
    binary = (mask > 127).astype(np.uint8)

    # Separate touching cracks into individual instances
    num_labels, labels_map = cv2.connectedComponents(binary)

    label_lines = []
    for label_id in range(1, num_labels):      # 0 = background
        instance = (labels_map == label_id).astype(np.uint8)

        if instance.sum() < MIN_AREA:
            continue

        # Find contours for this instance
        contours, _ = cv2.findContours(
            instance, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )

        for contour in contours:
            if len(contour) < MIN_POINTS // 2:
                continue

            # Flatten and normalize to [0, 1]
            pts = contour.squeeze()
            if pts.ndim == 1:
                pts = pts.reshape(1, 2)

            # Simplify contour slightly to reduce file size
            epsilon = 0.002 * cv2.arcLength(contour, True)
            simplified = cv2.approxPolyDP(contour, epsilon, True).squeeze()
            if simplified.ndim == 1:
                simplified = simplified.reshape(1, 2)
            if len(simplified) < 3:
                simplified = pts

            norm = []
            for x, y in simplified:
                norm.append(x / img_w)
                norm.append(y / img_h)

            if len(norm) < MIN_POINTS:
                continue

            coords_str = " ".join(f"{v:.6f}" for v in norm)
            label_lines.append(f"{CLASS_ID} {coords_str}")

    return label_lines


def process_split(src_dir: Path, dst_dir: Path, split_name: str):
    """Process uncropped val or test split."""
    split_dir = src_dir / f"{split_name}data"
    if not split_dir.exists():
        print(f"  [Warning] Directory {split_dir} does not exist, skipping split {split_name}.")
        return 0

    dst_img_dir = dst_dir / "images" / split_name
    dst_lbl_dir = dst_dir / "labels" / split_name

    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lbl_dir.mkdir(parents=True, exist_ok=True)

    # Find all image files (jpg/jpeg/png that do not contain '_mask')
    all_files = sorted(split_dir.iterdir())
    image_files = [
        f for f in all_files 
        if f.suffix.lower() in ('.jpg', '.jpeg', '.png')
        and '_mask' not in f.name.lower()
        and ':Zone.Identifier' not in f.name
    ]

    converted = 0
    skipped = 0

    for img_path in tqdm(image_files, desc=f"  {split_name}", leave=False):
        stem = img_path.stem

        # Find mask (stem + "_mask.png")
        mask_path = split_dir / f"{stem}_mask.png"
        if not mask_path.exists():
            skipped += 1
            continue

        # Read image to get dimensions (matches how cv2.imread auto-rotates it based on EXIF)
        img = cv2.imread(str(img_path))
        if img is None:
            skipped += 1
            continue
        h, w = img.shape[:2]

        # Get EXIF rotation from image
        exif_orientation = get_exif_rotation(img_path)

        # Convert mask to YOLO seg labels (rotating it to match)
        label_lines = binary_mask_to_yolo_instances(str(mask_path), w, h, exif_orientation)

        # Copy image
        dst_img_path = dst_img_dir / img_path.name
        shutil.copy2(img_path, dst_img_path)

        # Write label file (even if empty — YOLO needs it)
        dst_lbl_path = dst_lbl_dir / f"{stem}.txt"
        with open(dst_lbl_path, "w") as f:
            f.write("\n".join(label_lines))

        converted += 1

    print(f"  {split_name}: {converted} images converted, {skipped} skipped")
    return converted


def main():
    parser = argparse.ArgumentParser(description="Convert Crack500 Uncropped splits to YOLO seg format")
    parser.add_argument(
        "--src",
        type=str,
        default="data/datasets/crack500",
        help="Path to crack500 root dir"
    )
    parser.add_argument(
        "--dst",
        type=str,
        default="data/datasets/crack500_uncropped_yolo",
        help="Output directory"
    )
    args = parser.parse_args()

    src = Path(args.src).expanduser().resolve()
    dst = Path(args.dst).expanduser().resolve()

    print(f"[Convert] Source: {src}")
    print(f"[Convert] Output: {dst}")
    print()

    if not src.exists():
        print(f"ERROR: Source directory not found: {src}")
        return

    if dst.exists():
        print(f"[Warning] Output directory exists, clearing: {dst}")
        shutil.rmtree(dst)

    dst.mkdir(parents=True, exist_ok=True)

    counts = {}
    for split in ["val", "test"]:
        n = process_split(src, dst, split)
        counts[split] = n

    # Write dataset_uncropped.yaml
    yaml_content = f"""# Crack500 Uncropped — YOLO seg format
# Auto-generated by convert_crack500_uncropped.py

path: {dst.resolve()}
train: images/val
val:   images/val
test:  images/test

nc: 1
names:
  0: crack

# Stats
# val:   ~{counts.get('val', 0)} images (uncropped)
# test:  ~{counts.get('test', 0)} images (uncropped)
"""
    with open(dst / "dataset.yaml", "w") as f:
        f.write(yaml_content)
    print(f"\n  dataset.yaml written to {dst / 'dataset.yaml'}")
    print(f"[Done] Converted uncropped splits successfully.")


if __name__ == "__main__":
    main()


Writing scripts/convert_crack500_uncropped.py


In [3]:
# ── Step 1: Link / Prepare Uncropped Dataset ──
input_dir = Path("/kaggle/input/distill_datasetforme")
if not input_dir.exists():
    input_dir = Path("/kaggle/input")

uncropped_dir = Path("data/datasets/crack500_uncropped_yolo")
uncropped_dir.mkdir(parents=True, exist_ok=True)

# Link raw crack500 if uncropped yolo not already converted
for root, dirs, files in os.walk(str(input_dir)):
    root_p = Path(root)
    if "valdata" in dirs or "testdata" in dirs:
        !python scripts/convert_crack500_uncropped.py --src {root_p} --dst data/datasets/crack500_uncropped_yolo
        break

print("Uncropped validation dataset ready at data/datasets/crack500_uncropped_yolo/")


[Convert] Source: /kaggle/input/datasets/rauffatali/distill-datasetforme/datasets/crack500
[Convert] Output: /kaggle/working/data/datasets/crack500_uncropped_yolo

[Warning] Output directory exists, clearing: /kaggle/working/data/datasets/crack500_uncropped_yolo
  val: 50 images converted, 0 skipped
  test: 200 images converted, 0 skipped

  dataset.yaml written to /kaggle/working/data/datasets/crack500_uncropped_yolo/dataset.yaml
[Done] Converted uncropped splits successfully.
Uncropped validation dataset ready at data/datasets/crack500_uncropped_yolo/


In [4]:
# ── Step 2: Gaussian-Weighted Tiled Sliding-Window Inference Engine ──
import torch

def create_gaussian_weight_map(tile_size=512, sigma=0.35):
    """Generates a 2D Gaussian window to smoothly blend overlapping tiles."""
    ax = np.linspace(-1, 1, tile_size)
    gauss_1d = np.exp(-0.5 * (ax / sigma) ** 2)
    gauss_2d = np.outer(gauss_1d, gauss_1d).astype(np.float32)
    gauss_2d = np.maximum(gauss_2d, 0.05)  # minimum baseline floor for edges
    return gauss_2d / gauss_2d.max()


def tiled_predict_image_gaussian(model, img_bgr, tile_size=512, stride=384, conf=0.25, sigma=0.35):
    """Runs overlapping inference on full-res image with 2D Gaussian apodization blending."""
    h, w = img_bgr.shape[:2]
    full_prob_map = np.zeros((h, w), dtype=np.float32)
    weight_accum_map = np.zeros((h, w), dtype=np.float32)
    weight_window = create_gaussian_weight_map(tile_size, sigma)
    
    y_steps = list(range(0, max(1, h - tile_size + 1), stride))
    if y_steps[-1] + tile_size < h:
        y_steps.append(h - tile_size)
        
    x_steps = list(range(0, max(1, w - tile_size + 1), stride))
    if x_steps[-1] + tile_size < w:
        x_steps.append(w - tile_size)
        
    for y0 in y_steps:
        for x0 in x_steps:
            tile = img_bgr[y0:y0+tile_size, x0:x0+tile_size]
            results = model.predict(tile, imgsz=tile_size, conf=conf, verbose=False, device="cuda" if torch.cuda.is_available() else "cpu")
            r = results[0]
            
            tile_prob = np.zeros((tile_size, tile_size), dtype=np.float32)
            if r.masks is not None and len(r.masks) > 0:
                for m in r.masks.data.cpu().numpy():
                    m_resized = cv2.resize(m, (tile_size, tile_size))
                    tile_prob = np.maximum(tile_prob, m_resized)
                    
            full_prob_map[y0:y0+tile_size, x0:x0+tile_size] += tile_prob * weight_window
            weight_accum_map[y0:y0+tile_size, x0:x0+tile_size] += weight_window
            
    weight_accum_map = np.maximum(weight_accum_map, 1e-5)
    full_prob_map = full_prob_map / weight_accum_map
    return (full_prob_map > 0.35).astype(np.uint8)

print("Gaussian-weighted tiled inference engine ready!")


Gaussian-weighted tiled inference engine ready!


In [5]:
# ── Step 3: Discover Checkpoints & Run Cross-Evaluation ──
import os, zipfile, glob, shutil
from pathlib import Path
from tqdm import tqdm
import cv2
import numpy as np
import json
from ultralytics import YOLO

# ── Self-Contained Helper Functions ──
def compute_dice(pred_mask, gt_mask):
    intersection = np.logical_and(pred_mask, gt_mask).sum()
    total = pred_mask.sum() + gt_mask.sum()
    if total == 0:
        return 1.0 if intersection == 0 else 0.0
    return float(2.0 * intersection / total)

def create_gaussian_weight_map(tile_size=512, sigma=0.35):
    ax = np.linspace(-1, 1, tile_size)
    gauss_1d = np.exp(-0.5 * (ax / sigma) ** 2)
    gauss_2d = np.outer(gauss_1d, gauss_1d).astype(np.float32)
    gauss_2d = np.maximum(gauss_2d, 0.05)
    return gauss_2d / gauss_2d.max()

def predict_sliding_window_gaussian(model, img_bgr, tile_size=512, overlap=0.25, conf=0.25, sigma=0.35):
    h, w = img_bgr.shape[:2]
    stride = int(tile_size * (1.0 - overlap))
    full_prob_map = np.zeros((h, w), dtype=np.float32)
    weight_accum_map = np.zeros((h, w), dtype=np.float32)
    weight_window = create_gaussian_weight_map(tile_size, sigma)
    
    y_steps = list(range(0, max(1, h - tile_size + 1), stride))
    if len(y_steps) == 0 or (y_steps[-1] + tile_size < h):
        y_steps.append(max(0, h - tile_size))
        
    x_steps = list(range(0, max(1, w - tile_size + 1), stride))
    if len(x_steps) == 0 or (x_steps[-1] + tile_size < w):
        x_steps.append(max(0, w - tile_size))
        
    for y0 in y_steps:
        for x0 in x_steps:
            tile = img_bgr[y0:y0+tile_size, x0:x0+tile_size]
            actual_th, actual_tw = tile.shape[:2]
            if actual_th != tile_size or actual_tw != tile_size:
                tile_padded = np.zeros((tile_size, tile_size, 3), dtype=np.uint8)
                tile_padded[:actual_th, :actual_tw] = tile
                tile = tile_padded
                
            results = model.predict(tile, imgsz=tile_size, conf=conf, verbose=False)[0]
            tile_prob = np.zeros((tile_size, tile_size), dtype=np.float32)
            
            if results.masks is not None and len(results.masks) > 0:
                for m in results.masks.data.cpu().numpy():
                    m_resized = cv2.resize(m, (tile_size, tile_size))
                    tile_prob = np.maximum(tile_prob, m_resized)
                    
            actual_h = min(tile_size, h - y0)
            actual_w = min(tile_size, w - x0)
            
            full_prob_map[y0:y0+actual_h, x0:x0+actual_w] += tile_prob[:actual_h, :actual_w] * weight_window[:actual_h, :actual_w]
            weight_accum_map[y0:y0+actual_h, x0:x0+actual_w] += weight_window[:actual_h, :actual_w]
            
    weight_accum_map = np.maximum(weight_accum_map, 1e-5)
    full_prob_map = full_prob_map / weight_accum_map
    return (full_prob_map > 0.35).astype(np.uint8)

print("[Step 3] Scanning /kaggle/input for checkpoints...")

repack_dir = Path("/kaggle/working/repacked_ckpts")
repack_dir.mkdir(parents=True, exist_ok=True)

# A. Handle unzipped PyTorch model folders (contains data.pkl like bestmosaic/best)
if os.path.exists("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        if "data.pkl" in files:
            folder_name = os.path.basename(root)
            if folder_name in [".", ""]:
                folder_name = os.path.basename(os.path.dirname(root))
            repacked_path = repack_dir / f"{folder_name}.pt"
            print(f"  -> Detected unzipped PyTorch folder: {root}")
            print(f"  -> Repacking into valid PyTorch container: {repacked_path}...")
            with zipfile.ZipFile(repacked_path, 'w', compression=zipfile.ZIP_STORED) as zf:
                for r, d, f_list in os.walk(root):
                    for file_name in f_list:
                        full_file = os.path.join(r, file_name)
                        rel_file = os.path.relpath(full_file, root)
                        arc_name = os.path.join("archive", rel_file).replace("\\", "/")
                        zf.write(full_file, arc_name)
            print(f"  -> Successfully created valid model: {repacked_path}")

        # B. Handle any .zip archives (extract & check)
        for f in files:
            if f.endswith(".zip") and "crack500" not in f.lower():
                full_path = os.path.join(root, f)
                out_unzip = Path("/kaggle/working/unzipped_ckpts") / Path(f).stem
                out_unzip.mkdir(parents=True, exist_ok=True)
                try:
                    with zipfile.ZipFile(full_path, 'r') as zf:
                        zf.extractall(out_unzip)
                    print(f"  -> Extracted zip {f} to: {out_unzip}")
                except Exception as e:
                    print(f"  -> Failed to unzip {f}: {e}")

# Discover all .pt checkpoints (deduplicated by resolved absolute path)
ckpts_dict = {}
search_roots = ["/kaggle/working/repacked_ckpts", "/kaggle/input", "/kaggle/working", "runs"]
for s_root in search_roots:
    if os.path.exists(s_root):
        for root, dirs, files in os.walk(s_root):
            for f in files:
                if (f.endswith(".pt") or f.endswith(".pth")) and not f.startswith("yolo11n"):
                    resolved = str(Path(os.path.join(root, f)).resolve())
                    if resolved not in ckpts_dict:
                        ckpts_dict[resolved] = Path(resolved)

ckpts = list(ckpts_dict.values())
print(f"\n[Step 3] Discovered {len(ckpts)} model checkpoints:")
for c in ckpts:
    print(f"  - {c}")

if len(ckpts) == 0:
    print("\n[WARNING] No .pt model files found.")

# Pre-index ALL ground-truth masks across /kaggle/input and data directories
print("\n[Step 3] Indexing ground-truth mask database...")
mask_database = {}
for search_dir in ["/kaggle/input", "/kaggle/working", "data"]:
    if os.path.exists(search_dir):
        for root, dirs, files in os.walk(search_dir):
            if "teacher_logits" in root:
                continue
            for f in files:
                if f.endswith(".png") or f.endswith("_mask.png"):
                    clean_stem = f.replace("_mask.png", "").replace(".png", "")
                    mask_database[clean_stem] = os.path.join(root, f)

print(f"  -> Indexed {len(mask_database)} ground-truth mask files.")

# Find ground-truth uncropped images
val_img_dir = Path("data/datasets/crack500_uncropped_yolo/images/val")
all_val_imgs = sorted(list(val_img_dir.glob("*.jpg")) + list(val_img_dir.glob("*.png")))
print(f"  -> Found {len(all_val_imgs)} uncropped validation images.")

eval_summary = {}
for ckpt in ckpts:
    name = ckpt.parent.parent.name if ckpt.parent.name in ["weights", "repacked_ckpts"] else ckpt.stem
    if name == "repacked_ckpts" or name == "working":
        name = ckpt.stem
    print(f"\n{'='*50}\nEvaluating Checkpoint: {name}\nPath: {ckpt}\n{'='*50}")
    model = YOLO(str(ckpt))
    
    # 1. Direct Resize Val (512x512)
    res_direct = model.val(data="data/datasets/crack500_uncropped_yolo/dataset.yaml", split="val", verbose=False)
    direct_mAP50 = float(res_direct.seg.map50)
    direct_mAP50_95 = float(res_direct.seg.map)
    direct_box_mAP50 = float(res_direct.box.map50)
    
    # 2. Tiled Sliding-Window Full-Resolution Dice Evaluation
    tiled_dices = []
    direct_dices = []
    
    for img_p in tqdm(all_val_imgs, desc=f"  Full-Res Tiled Eval ({name[:15]})", leave=False):
        img_bgr = cv2.imread(str(img_p))
        if img_bgr is None:
            continue
        h, w = img_bgr.shape[:2]
        
        # Fast lookup in pre-indexed mask database
        gt_mask_path = mask_database.get(img_p.stem)
        if gt_mask_path and os.path.exists(gt_mask_path):
            gt_mask = cv2.imread(gt_mask_path, cv2.IMREAD_GRAYSCALE)
            if gt_mask is not None:
                if gt_mask.shape[:2] != (h, w):
                    gt_mask = cv2.resize(gt_mask, (w, h), interpolation=cv2.INTER_NEAREST)
                gt_binary = (gt_mask > 127).astype(np.uint8)
                
                # A. Direct resize prediction
                r_dir = model.predict(img_bgr, imgsz=512, conf=0.25, verbose=False)[0]
                if r_dir.masks is not None and len(r_dir.masks) > 0:
                    pred_dir_mask = (r_dir.masks.data.cpu().numpy().max(axis=0) > 0.5).astype(np.uint8)
                    pred_dir_full = cv2.resize(pred_dir_mask, (w, h), interpolation=cv2.INTER_NEAREST)
                else:
                    pred_dir_full = np.zeros((h, w), dtype=np.uint8)
                direct_dices.append(compute_dice(pred_dir_full, gt_binary))
                
                # B. Tiled Sliding Window prediction with Gaussian apodization
                pred_tiled = predict_sliding_window_gaussian(model, img_bgr, tile_size=512, overlap=0.25, conf=0.25)
                tiled_dices.append(compute_dice(pred_tiled, gt_binary))
                
    mean_tiled_dice = float(np.mean(tiled_dices)) if tiled_dices else 0.0
    mean_direct_dice = float(np.mean(direct_dices)) if direct_dices else 0.0
    dice_boost = ((mean_tiled_dice - mean_direct_dice) / max(mean_direct_dice, 1e-6)) * 100
    
    print(f"\nLeaderboard Results for [{name}]:")
    print(f"  ├── Direct Resize (512x512) -> Mask mAP50: {direct_mAP50:.4f}, Mask mAP50-95: {direct_mAP50_95:.4f}, Box mAP50: {direct_box_mAP50:.4f}")
    print(f"  └── Full-Res Megapixel Dice -> Direct: {mean_direct_dice:.4f} | Tiled Gaussian: {mean_tiled_dice:.4f} (+{dice_boost:.1f}% boost)")
    
    eval_summary[name] = {
        "checkpoint_path": str(ckpt),
        "direct_mask_mAP50": direct_mAP50,
        "direct_mask_mAP50_95": direct_mAP50_95,
        "direct_box_mAP50": direct_box_mAP50,
        "mean_direct_dice": mean_direct_dice,
        "mean_tiled_dice": mean_tiled_dice,
        "dice_boost_percent": dice_boost,
    }

out_path = Path("/kaggle/working/results/ood_eval_summary.json")
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, "w") as f:
    json.dump(eval_summary, f, indent=2)
print(f"\nSaved evaluation summary to {out_path}")


[Step 3] Scanning /kaggle/input for checkpoints...
  -> Detected unzipped PyTorch folder: /kaggle/input/datasets/shahinalakparov/bestmosaic/best
  -> Repacking into valid PyTorch container: /kaggle/working/repacked_ckpts/best.pt...
  -> Successfully created valid model: /kaggle/working/repacked_ckpts/best.pt

[Step 3] Discovered 1 model checkpoints:
  - /kaggle/working/repacked_ckpts/best.pt

[Step 3] Indexing ground-truth mask database...
  -> Indexed 4140 ground-truth mask files.
  -> Found 50 uncropped validation images.

Evaluating Checkpoint: best
Path: /kaggle/working/repacked_ckpts/best.pt
Ultralytics 8.4.125 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
YOLO11n-seg summary (fused): 129 layers, 3,006,091 parameters, 0 gradients, 9.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2705.1±821.8 MB/s, size: 1915.2 KB)
val: Scanning /kaggle/working/data/datasets/crack500_uncropped_yolo/labels/val... 50 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5


Leaderboard Results for [best]:
  ├── Direct Resize (512x512) -> Mask mAP50: 0.1409, Mask mAP50-95: 0.0373, Box mAP50: 0.1747
  └── Full-Res Megapixel Dice -> Direct: 0.1651 | Tiled Gaussian: 0.2515 (+52.3% boost)

Saved evaluation summary to /kaggle/working/results/ood_eval_summary.json
